In [17]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [18]:
import os
os.environ["PINECONE_API_KEY"] = "pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ"
print("Pinecone API Key:", os.getenv("PINECONE_API_KEY"))
from Search_or_Discuss import classify_and_extract
from data_utils import fetch_papers, verify_results, chunk_papers
from Database import create_index, save_embeddings_to_index, delete_index
from retrieval import answer_query, clear_memory
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))




Pinecone API Key: pcsk_594Xgb_Qtwvagd2kRcdaJtLSdnXXrnn2RN81sRkGZyJYCqQmiEJY1iw6rDdNATRvSmfkCZ


In [19]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

def handleUserQuery(user_input, user_index):
    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)

    if classification["action"] == "search":
        filters = classification['filters']
        print("\nRunning Search with filters:", filters)

        # Step 1: Fetch papers
        papers = fetch_papers(filters)

        # Step 2: Verify & Filter papers
        filtered_papers = verify_results(papers, filters)

        # Step 3: Chunk papers
        paper_chunks = chunk_papers(filtered_papers)

        text = []
        # Print sample chunks and save embeddings (update as needed)
        for i, chunk in enumerate(paper_chunks):
            print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")
            text.append(chunk.page_content)
        save_embeddings_to_index(index_name=user_index, text=text)

        question = classification['question']
        answer = answer_query(question, index_obj, top_k=3)
        print("Generated Answer:")
        print(answer)

    elif classification["action"] == "discuss":
        question = classification['question']
        print("\nDiscussion Topic:", question)
        # Call the RAG pipeline to generate a discussion answer
        answer = answer_query(question, index_obj, top_k=3)
        print("Generated Answer:")
        print(answer)
    else:
        # Default behavior if classification doesn't match expected actions:
        print("Action not recognized. Please try again.")

# Use a loop to enable continuous conversation
while True:
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        break
    handleUserQuery(user_query, index_name)
delete_index(index_name=index_name)
clear_memory()

Index 'gd3toivjr6' created successfully!

Running Search with filters: {'query': 'typiclust', 'author': None, 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '10', 'sort_by': 'newest'}

Chunk 1:
ICLR 2024 Workshop on Practical Machine Learning for Low Resource Settings
BRIDGING DIVERSITY AND UNCERTAINTY IN ACTIVE
LEARNING WITH SELF -SUPERVISED PRE-TRAINING
Paul Doucet, Benjamin Estermann, Till Aczel & Roger Wattenhofer
ETH Z¨urich
{pdoucet,estermann,taczel,wattenhofer}@ethz.ch
ABSTRACT
This...

Chunk 2:
supervised pre-trained models. We introduce a straightforward heuristic called
TCM that mitigates the cold start problem while maintaining strong performance
across various data levels. By initially applying TypiClust for diversity sampling
and subsequently transitioning to uncertainty sampling with...

Chunk 3:
in both low and high data regimes.
1 I NTRODUCTION
Training mach

PineconeApiException: (400)
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Date': 'Wed, 02 Apr 2025 14:21:51 GMT', 'Content-Type': 'application/json', 'Content-Length': '74', 'Connection': 'keep-alive', 'x-pinecone-request-latency-ms': '25', 'x-pinecone-request-id': '4287018082823170735', 'x-envoy-upstream-service-time': '26', 'server': 'envoy'})
HTTP response body: {"code":3,"message":"No vectors provided for upsert request","details":[]}
